In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

model = AutoModel.from_pretrained("facebook/mms-tts-mal")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-mal")

In [3]:
import os
import pandas as pd
import torchaudio
import librosa
import numpy as np
from jiwer import wer, cer

metadata = pd.read_csv("metadata_malayalam.csv")
print(metadata)

                           ID  \
0   train_malayalammale_00001   
1   train_malayalammale_00002   
2   train_malayalammale_00003   
3   train_malayalammale_00004   
4   train_malayalammale_00005   
5   train_malayalammale_00006   
6   train_malayalammale_00007   
7   train_malayalammale_00008   
8   train_malayalammale_00009   
9   train_malayalammale_00010   
10  train_malayalammale_00011   
11  train_malayalammale_00012   
12  train_malayalammale_00013   
13  train_malayalammale_00014   
14  train_malayalammale_00015   
15  train_malayalammale_00016   
16  train_malayalammale_00017   
17  train_malayalammale_00018   
18  train_malayalammale_00019   
19  train_malayalammale_00020   
20  train_malayalammale_00021   
21  train_malayalammale_00022   
22  train_malayalammale_00023   
23  train_malayalammale_00024   
24  train_malayalammale_00025   
25  train_malayalammale_00026   
26  train_malayalammale_00027   
27  train_malayalammale_00028   
28  train_malayalammale_00029   
29  train_

In [4]:
output_folder = "generated_wavs_malayalam"
os.makedirs(output_folder, exist_ok=True)

In [6]:
import scipy
# Iterate through each row of the metadata to generate and save audio
for index, row in metadata.iterrows():
    text = row['Text']
    wav_name = f"{row['ID']}.wav"  # Add the .wav extension
    output_path = os.path.join(output_folder, wav_name)

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt")

    # Generate the audio waveform
    with torch.no_grad():
        output = model(**inputs).waveform

    # Save the generated audio as a .wav file
    scipy.io.wavfile.write(output_path, rate=model.config.sampling_rate, data=output.squeeze().numpy())

    print(f"Generated and saved: {output_path}")

Generated and saved: generated_wavs_malayalam\train_malayalammale_00001.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00002.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00003.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00004.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00005.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00006.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00007.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00008.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00009.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00010.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00011.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00012.wav
Generated and saved: generated_wavs_malayalam\train_malayalammale_00013.wav
Generated an

In [7]:
import os
import csv
import librosa
import numpy as np
from jiwer import cer, wer
import parselmouth

def compute_metrics(original_folder, generated_folder, output_csv):
    metrics = {
        "File": [], "MCD": [], "LSD": [], "SNR": [],
        "Pitch RMSE": [], "Duration Difference": [], "CER": [], "WER": []
    }
    
    original_files = sorted(os.listdir(original_folder))
    generated_files = sorted(os.listdir(generated_folder))
    
    for orig_file, gen_file in zip(original_files, generated_files):
        orig_path = os.path.join(original_folder, orig_file)
        gen_path = os.path.join(generated_folder, gen_file)
        
        # Load audio files
        orig_audio, orig_sr = librosa.load(orig_path, sr=None)
        gen_audio, gen_sr = librosa.load(gen_path, sr=None)
        duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        
        # Resample if needed
        if orig_sr != gen_sr:
            gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
            gen_sr = orig_sr
        
        # Align audio lengths
        min_length = min(len(orig_audio), len(gen_audio))
        orig_audio = orig_audio[:min_length]
        gen_audio = gen_audio[:min_length]
        
        # Compute spectrograms
        orig_mel = librosa.feature.melspectrogram(y=orig_audio, sr=orig_sr)
        gen_mel = librosa.feature.melspectrogram(y=gen_audio, sr=gen_sr)
        
        # Align spectrogram shapes
        min_frames = min(orig_mel.shape[1], gen_mel.shape[1])
        orig_mel = orig_mel[:, :min_frames]
        gen_mel = gen_mel[:, :min_frames]
        
        # Compute metrics
        mcd = np.mean(np.abs(orig_mel - gen_mel))  # Simplified
        lsd = np.mean(np.abs(librosa.amplitude_to_db(orig_mel) - librosa.amplitude_to_db(gen_mel)))
        noise = orig_audio - gen_audio
        snr = 10 * np.log10(np.sum(orig_audio ** 2) / np.sum(noise ** 2))
        orig_pitch = parselmouth.Sound(orig_path).to_pitch().selected_array["frequency"]
        gen_pitch = parselmouth.Sound(gen_path).to_pitch().selected_array["frequency"]
        min_pitch_length = min(len(orig_pitch), len(gen_pitch))
        pitch_rmse = np.sqrt(np.mean((orig_pitch[:min_pitch_length] - gen_pitch[:min_pitch_length]) ** 2))
        # duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        orig_text = os.path.splitext(orig_file)[0]  # Assumes filename contains transcription
        gen_text = os.path.splitext(gen_file)[0]  # Assumes filename contains transcription
        char_error_rate = cer(orig_text, gen_text)
        word_error_rate = wer(orig_text, gen_text)
        
        # Append to metrics
        metrics["File"].append(orig_file)
        metrics["MCD"].append(mcd)
        metrics["LSD"].append(lsd)
        metrics["SNR"].append(snr)
        metrics["Pitch RMSE"].append(pitch_rmse)
        metrics["Duration Difference"].append(duration_diff)
        metrics["CER"].append(char_error_rate)
        metrics["WER"].append(word_error_rate)
    
    # Write metrics to a CSV file
    with open(output_csv, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(metrics.keys())  # Write header
        writer.writerows(zip(*metrics.values()))  # Write rows
    
    print(f"Metrics saved to {output_csv}")



# Folders containing original and generated wav files
original_folder = "D:/Wav2Lip-master/TTS Evaluation/wav_malayalam"
generated_folder = "D:/Wav2Lip-master/TTS Evaluation/generated_wavs_malayalam"

# Output CSV file
output_csv = "tts_model_evaluation_malayalam.csv"

# Compute and save metrics
compute_metrics(original_folder, generated_folder, output_csv)


C:\Users\satvi\AppData\Local\Temp\ipykernel_23360\1874766856.py:28: FutureWarning: Pass orig_sr=16000, target_sr=48000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
C:\Users\satvi\AppData\Local\Temp\ipykernel_23360\1874766856.py:28: FutureWarning: Pass orig_sr=16000, target_sr=48000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)


Metrics saved to tts_model_evaluation_malayalam.csv
